In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/cs1851"

/content/drive/MyDrive/Colab Notebooks/cs1851


In [ ]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from random_forest_pipeline import load_training_data, fit_preprocessors, transform_with_preprocessors, make_train_val_split, build_random_forest, tune_random_forest, evaluate_model, load_test_data, make_submission
from cnn_pipeline import CNN, FCN, train_one_epoch, run_experiment, evaluate, ImageDataset, build_dataloader, predict_test, build_testdata


## Data + Setup

In [ ]:
# generic path
path = "/content/drive/MyDrive/"
data_path = "Colab Notebooks/cs1851/data/"

########### MODEL RESULT SAVE PATHS ###########
rf_file_path = "Colab Notebooks/cs1851/rf_val_predictions.csv"
train_rf = "Colab Notebooks/cs1851/rf_train_predictions.csv"
rf_save_path = os.path.join(os.path.dirname(path), rf_file_path)
train_rf_save_path = os.path.join(os.path.dirname(path), train_rf)
cnn_file_path = "Colab Notebooks/cs1851/cnn_val_predictions.csv"
cnn_save_path = os.path.join(os.path.dirname(path), cnn_file_path)

torch.manual_seed(32)
np.random.seed(32)

EPOCHS = 70
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #this ensures that the code runs on GPUs if available
print(f"Device: {DEVICE}\n")

Device: cpu



In [ ]:
# edit data_path to point to data
images = np.load(path + data_path + "train_images.npy")
labels = np.load(path + data_path + "train_labels.npy")
ids = np.load(path + data_path + "train_ids.npy", allow_pickle=True)

# Image dataset (cnn_pipeline)
train_loader, val_loader = build_dataloader(images, labels, ids)

# Tabular data (from random_forest_pipeline)
X_raw, y, ids = load_training_data(data_dir=path + data_path)
X_train_raw, X_val_raw, y_train, y_val, rf_train_ids, rf_val_ids = make_train_val_split(
        X_raw, y, ids, test_size=0.2, random_state=42
    )
X_train, age_median, sex_encoder, site_encoder = fit_preprocessors(X_train_raw)
X_val = transform_with_preprocessors(X_val_raw, age_median, sex_encoder, site_encoder)
print("Processed X_train shape:", X_train.shape)
print("Processed X_val shape:", X_val.shape)

Processed X_train shape: (2800, 3)
Processed X_val shape: (700, 3)


## Train Models

- Trains image (CNN + FCN) and tabular (RF) models
- saves model weights and validation results (ids, class probabilities per example, true labels, and class predictions)

### Tabular Model (RF)

In [ ]:
def train_rf(X_train, y_train):
  '''tunes against f1 macro, trains rf,  and saves the trained model and processors to rf_pipeline.pkl'''

  rf, best_params, best_score = tune_random_forest(X_train, y_train)
  rf.fit(X_train, y_train)
  joblib.dump({
      "model": rf,
      "age_median": age_median,
      "sex_encoder": sex_encoder,
      "site_encoder": site_encoder
  }, "rf_pipeline.pkl")
  return rf

def eval_rf(rf, X_val, y_val, val_ids):
  '''evaluates the model and saves outputs to a dataframe'''
  metrics, y_pred, y_proba = evaluate_model(rf, X_val, y_val)

  val_results = pd.DataFrame({
        "ID": val_ids,
        "true_label": y_val,
        "rf_pred": y_pred
    })
  for i in range(y_proba.shape[1]):
    val_results[f"rf_prob_{i}"] = y_proba[:, i]

  # saving results to csv (for now)
  # change file_path to specific location
  return df, y_pred, y_proba



Fitting 3 folds for each of 162 candidates, totalling 486 fits

Best Parameters
   max_depth max_features  min_samples_leaf  min_samples_split  n_estimators
0         10         sqrt                 1                 10           300

Best CV Macro F1
   f1_macro_cv
0       0.2186

Validation Metrics
   accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
0    0.5086           0.2179        0.2362    0.2162       0.5606

Classification Report
              precision    recall  f1-score   support

           0       0.18      0.22      0.20        77
           1       0.18      0.24      0.21        41
           2       0.12      0.33      0.18        24
           3       0.15      0.19      0.17        77
           4       0.89      0.66      0.76       463
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00        10

    accuracy                           0.51       700
   macro avg       0.22      0.24      0.22       700
w

In [ ]:
# save validation results
rf = train_rf(X_train, y_train)
y_proba = rf.predict_proba(X_train)
rf_val_results, _, _ = eval_rf(rf, X_val, y_val, rf_val_ids)

rf_val_results.to_csv(rf_save_path, index=False)
print(f"\nSaved validation probabilities to: {rf_save_path}")

# saves train results for later fusion
train_rf_df = pd.DataFrame({"ID": rf_train_ids,
        "true_label": y_train})

for i in range(y_proba.shape[1]):
    train_rf_df[f"rf_prob_{i}"] = y_proba[:, i]

train_rf_df.to_csv(train_rf_save_path, index=False)
print(f"\nSaved train probabilities to: {train_rf_save_path}")

###CNN

In [ ]:
def train_cnn(train_loader, val_loader, device):

  cnn = CNN(device=device, num_classes=7).to(device)
  hist_base, val_info  = run_experiment(cnn, train_loader, val_loader, EPOCHS, lr=1e-4)
  return val_info

def format_cnn_outputs(val_info):
  probs, preds, labels, ids = val_info

  val_results = pd.DataFrame({
        "ID": ids,
        "true_label": labels,
        "cnn_pred": preds
    })
  for i in range(probs.shape[1]):
    val_results[f"cnn_prob_{i}"] = probs[:, i]
  return val_results
# criterion = nn.CrossEntropyLoss()
# val_loss, macro_f1, eval_metrics = evaluate(cnn, val_loader, criterion)
# probs, preds, labels, ids

SyntaxError: expected ':' (2497007897.py, line 1)

In [ ]:
# cnn pipeline + save validation results in same format as rf
val_info = train_cnn(train_loader, val_loader, DEVICE)
cnn_val_results = format_cnn_outputs(val_info)

cnn_val_results.to_csv(cnn_save_path, index=False)
print(f"\nSaved validation probabilities to: {cnn_save_path}")

###FCN
- Not updated to be implemented with intermediate or late fusion

In [ ]:
criterion = nn.CrossEntropyLoss()

fcn = FCN(criterion, train_loader, val_loader, DEVICE, lr=1e-3).to(DEVICE)
hist_fully, _ ,val_loss, probs, preds, labels, ids = fcn.fit(epochs=EPOCHS)

Epoch 10 loss: 0.9378535815647671 val loss 1.006931678227016
Epoch 20 loss: 0.9132648951666695 val loss 0.917986981528146
Epoch 30 loss: 0.8832471820286342 val loss 0.9328027514048985
Epoch 40 loss: 0.8583925761495318 val loss 0.9596978391919817
Epoch 50 loss: 0.8469250896998815 val loss 0.9300201327460152
Epoch 60 loss: 0.810165605204446 val loss 0.9110370349884033
Epoch 70 loss: 0.7688422730990818 val loss 0.9078496210915702
Epoch 80 loss: 0.7255132978303092 val loss 0.9071518346241543
Epoch 90 loss: 0.6915364180292402 val loss 0.9241845968791417
Epoch 100 loss: 0.6730235859325954 val loss 0.9302720165252686
Epoch 100/100  |  Loss = 42.8288


In [ ]:
val_results = pd.DataFrame({
        "ID": ids,
        "true_label": labels,
        "fcn_pred": preds
    })
for c in range(probs.shape[1]):
    val_results[f"fcn_prob_{c}"] = probs[:, c]

file_path = "Colab Notebooks/cs1851/fcn_val_predictions.csv"
fcn_save_path = os.path.join(os.path.dirname(path), file_path)
val_results.to_csv(fcn_save_path, index=False)
print(f"\nSaved validation probabilities to: {fcn_save_path}")


Saved validation probabilities to: /content/drive/MyDrive/Colab Notebooks/cs1851/fcn_val_predictions.csv


## Intermediate Fusion

In [ ]:
def get_features(model, loader, device):
  '''extracts CNN feature embeddings to use for intermediate fusion
      optionally returns labels if provided by the dataloader'''
  model.eval()
  features = []
  labels = []
  all_ids = []

  with torch.no_grad():
    for i, batch in enumerate(loader):

      if len(batch) == 3:
        x, y, ids = batch
        labels.append(y.cpu().numpy())
      else:
        x, ids = batch
        y = None

      x = x.to(device)
      feats = model.forward_feats(x)

      features.append(feats.cpu().numpy())
      all_ids.extend(list(ids))

    features = np.concatenate(features, axis=0)
    if len(labels) > 0:
      labels = np.concatenate(labels, axis=0)
      return features, labels, np.array(all_ids)

    return features, np.array(all_ids)

In [ ]:
# load trained CNN and get feature embeddings for train/val for intermediate fusion

##### ONLY RERUN IF FEATURE FILE IS NOT SAVED (~ 11 minutes on cpu) ########
fus_cnn = CNN(device=DEVICE, num_classes=7).to(DEVICE)
fus_cnn.load_state_dict(torch.load("100epoch_cnn_best.pt", map_location=torch.device(DEVICE)))

Z_img_train, y_img_train, ids_train = get_features(fus_cnn, train_loader, DEVICE)
Z_img_val, y_img_val, ids_val = get_features(fus_cnn, val_loader, DEVICE)

np.savez(
    "cnn_features.npz",
    Z_img_train=Z_img_train,
    y_img_train=y_img_train,
    ids_train=ids_train,
    Z_img_val=Z_img_val,
    y_img_val=y_img_val,
    ids_val=ids_val
)

In [ ]:
def get_rf_feats(model, X_train, X_val):
  '''get intermediate rf features (leaf indices)'''
  Z_train = model.apply(X_train)
  Z_val = model.apply(X_val)

  return Z_train, Z_val


In [ ]:
from sklearn.decomposition import TruncatedSVD
# load CNN features and apply truncated svd to reduce dimensionality for fusion
def safe_k(X, k_wanted):
    # TruncatedSVD requires 1 <= k < n_features, and also k < n_samples in many practical cases
    n_samples, n_features = X.shape
    k_max = max(1, min(n_features - 1, n_samples - 1))
    return min(k_wanted, k_max)

def apply_svd( X_train, X_val, ids_train, ids_val, prefix, n_components=128):
  # train_leafs = model.apply(X_train)
  # val_leafs = model.apply(X_val)

  k = safe_k(X_train, n_components)
  tab_svd = TruncatedSVD(n_components=k, random_state=0)
  Z_svd_train = tab_svd.fit_transform(X_train)
  Z_svd_val = tab_svd.transform(X_val)

  cols = [f'{prefix}_{i}' for i in range(Z_svd_train.shape[1])]
  df_train = pd.DataFrame(Z_svd_train, columns=cols)
  df_train["ID"] = ids_train

  df_val = pd.DataFrame(Z_svd_val, columns=cols)
  df_val["ID"] = ids_val

  return df_train, df_val

def fuse_mid_features(rf_df, cnn_df):
    '''combines the tabular and image features by combining on ID'''
    print(len(rf_df) == len(cnn_df))

    merged = pd.merge(rf_df, cnn_df, on='ID')

    print(len(merged) == len(rf_df))
    X = merged.drop(columns=["ID"])

    return X, merged["ID"]


def intermediate_fusion(Z_train, y_train, Z_test, y_test=None):
  '''Performs intermediate fusion and optionally returns y labels for evaluation'''

  mid_clf = Pipeline([("scaler", StandardScaler()),
                  ("lr", LogisticRegression(max_iter=3000, class_weight='balanced'))])
  mid_clf.fit(Z_train, y_train)
  preds = mid_clf.predict(Z_test)


  if y_test is not None:
    print("F1:", f1_score(y_test, preds, average="macro"))
    print("accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))
    return preds, y_test

  else:
    return preds


## Late fusion
- (weighted) average fusion
- logistic fusion

In [ ]:
####### Average Fusion ###########

def get_probs(df, model_type):
  '''helper for avg_fusion'''
  probs = df[[f'{model_type}_prob_{i}' for i in range(7)]].values
  return probs

def avg_fusion(tab_df, img_df, tab_model, img_model):
    '''computes avgerage late fusion (optionally manually weighted)'''

    merged = pd.merge(tab_df, img_df, on="ID")
    # print(merged.columns)
    tab_probs = get_probs(merged, tab_model)
    img_probs = get_probs(merged, img_model)

    fusion_probs = 0.3 * tab_probs + 0.7 * img_probs
    fusion_preds = fusion_probs.argmax(axis=1)
    # print(fusion_preds)
    df = pd.DataFrame({"ID": merged["ID"],
                       "fusion_preds": fusion_preds})
    # print(list(merged.columns))
    if 'true_label_x' in list(merged.columns):
      y = merged['true_label_x'].values
      return y, df

    else:
      return df

In [ ]:
from sklearn.linear_model import LogisticRegression

############# LR Fusion #############
def train_lr_fusion(tab_df, img_df, tab_model, img_model):
    merged = pd.merge(tab_df, img_df, on="ID")

    tab_probs, img_probs = get_probs(merged, tab_model), get_probs(merged, img_model)

    features = np.concatenate([tab_probs, img_probs], axis=1)
    lr = LogisticRegression(class_weight='balanced', max_iter=3000)
    y = merged['true_label_x']
    lr.fit(features, y)
    return lr

def fit_lr_fusion(lr, tab_df, img_df, tab_model, img_model):

    merged = pd.merge(tab_df, img_df, on="ID")
    # print(list(merged.columns))

    tab_probs = get_probs(merged, tab_model)
    img_probs = get_probs(merged, img_model)

    features = np.concatenate([tab_probs, img_probs], axis=1)

    lr_probs = lr.predict_proba(features)
    preds = lr_probs.argmax(axis=1)

    df =  pd.DataFrame({
        "ID": merged["ID"],
        "fusion_preds": preds})

    if 'true_label_x' in list(merged.columns):
      y = merged['true_label_x'].values
      return y, df

    else:
      return df



## Validation Fusion Checks

###Late fusion
  1. (weighted) average
  2. logistic regression

In [ ]:
# load in validation dataframes for late fusion
val_rf_df = pd.read_csv(rf_save_path)
print(val_rf_df.columns)
val_cnn_df = pd.read_csv(cnn_save_path)
print(val_cnn_df.columns)
train_rf_df = pd.read_csv(train_rf_save_path)
# fcn_df = pd.read_csv(fcn_save_path)
# print(fcn_df.columns)

Index(['ID', 'true_label', 'rf_pred', 'rf_prob_0', 'rf_prob_1', 'rf_prob_2',
       'rf_prob_3', 'rf_prob_4', 'rf_prob_5', 'rf_prob_6'],
      dtype='object')
Index(['ID', 'true_label', 'cnn_pred', 'cnn_prob_0', 'cnn_prob_1',
       'cnn_prob_2', 'cnn_prob_3', 'cnn_prob_4', 'cnn_prob_5', 'cnn_prob_6'],
      dtype='object')


In [ ]:
print("Averaged Late Fusion")

y, df = avg_fusion(val_rf_df, val_cnn_df, 'rf', 'cnn')
print("RF + CNN accuracy:", accuracy_score(y, df['fusion_preds']))
print("RF + CNN f1 macro:", f1_score(y, df['fusion_preds'], average="macro"))
print(classification_report(y, df['fusion_preds']))

print('===========================================================')

print("Logistic Regression Late Fusion")

lr = train_lr_fusion(val_rf_df, val_cnn_df, 'rf', 'cnn')
y, lr_df = fit_lr_fusion(lr, val_rf_df, val_cnn_df, 'rf', 'cnn')
print("RF + CNN accuracy:", accuracy_score(y, lr_df["fusion_preds"]))
print("RF + CNN f1 macro:", f1_score(y, lr_df["fusion_preds"], average="macro"))
print(classification_report(y, lr_df['fusion_preds']))

Averaged Late Fusion
RF + CNN accuracy: 0.6542857142857142
RF + CNN f1 macro: 0.32484844045533917
              precision    recall  f1-score   support

           0       0.34      0.30      0.32        77
           1       0.38      0.15      0.21        41
           2       0.20      0.17      0.18        24
           3       0.32      0.38      0.35        77
           4       0.79      0.85      0.82       463
           5       0.00      0.00      0.00         8
           6       0.60      0.30      0.40        10

    accuracy                           0.65       700
   macro avg       0.37      0.31      0.32       700
weighted avg       0.63      0.65      0.64       700

Logistic Regression Late Fusion
RF + CNN accuracy: 0.6014285714285714
RF + CNN f1 macro: 0.35318001596773974
              precision    recall  f1-score   support

           0       0.37      0.44      0.40        77
           1       0.35      0.20      0.25        41
           2       0.17      0.58

###Intermediate fusion with baseline
  1. rf + cnn intermediate fusion features
  2. rf baseline
  3. cnn baseline

In [ ]:
rf_info = joblib.load("rf_pipeline.pkl")
rf_model = rf_info["model"]

data = np.load("cnn_features.npz", allow_pickle=True)
z_img_train = data["Z_img_train"]
z_img_val = data["Z_img_val"]

cnn_val_ids = data["ids_val"]
cnn_train_ids = data["ids_train"]
y_img_train = data["y_img_train"]
y_img_val = data["y_img_val"]

In [ ]:
######## INTERMEDIATE FUSION ###########
y_val = pd.Series(y_img_val, index=cnn_val_ids)
y_train = pd.Series(y_img_train, index=cnn_train_ids)

z_tab_train, z_tab_val = get_rf_feats(rf_model, X_train, X_val)

tab_df_train, tab_df_val = apply_svd(z_tab_train, z_tab_val, rf_train_ids, rf_val_ids, "rf_svd") # df with svd mid features and id
img_df_train, img_df_val = apply_svd(z_img_train, z_img_val, cnn_train_ids, cnn_val_ids, "cnn_svd")

Z_train, z_train_ids = fuse_mid_features(tab_df_train, img_df_train)
Z_val, z_val_ids = fuse_mid_features(tab_df_val, img_df_val)

# align labels with fusion ids
y_val_aligned = y_val.loc[z_val_ids].values
y_train_aligned = y_train.loc[z_train_ids].values

preds, y = intermediate_fusion(Z_train, y_train_aligned, Z_val, y_val_aligned)


True
True
True
True
F1: 0.32566099381045993
accuracy: 0.6185714285714285
              precision    recall  f1-score   support

           0       0.27      0.35      0.31        77
           1       0.31      0.44      0.36        41
           2       0.24      0.42      0.31        24
           3       0.27      0.35      0.31        77
           4       0.92      0.75      0.83       463
           5       0.00      0.00      0.00         8
           6       0.14      0.20      0.17        10

    accuracy                           0.62       700
   macro avg       0.31      0.36      0.33       700
weighted avg       0.70      0.62      0.65       700



In [ ]:
######## HYBRID INTERMEDIATE FUSION ###########
# load rf train/val metrics
val_rf_df = pd.read_csv(rf_save_path)
train_rf_df = pd.read_csv(train_rf_save_path)
val_probs = get_probs(val_rf_df, 'rf')
train_probs = get_probs(train_rf_df, 'rf')

# getting rf probs for hybrid (rf_late + cnn_mid)
cols = [f'rfprobs_{i}' for i in range(train_probs.shape[1])]
df_train = pd.DataFrame(train_probs, columns=cols)
df_train["ID"] = train_rf_df["ID"]

val_cols = [f'rfprobs_{i}' for i in range(val_probs.shape[1])]
df_val = pd.DataFrame(val_probs, columns=val_cols)
df_val["ID"] = val_rf_df["ID"]

Z_train, z_id_train = fuse_mid_features(df_train,  img_df_train)
Z_val, z_id_val = fuse_mid_features(df_val, img_df_val)

# print(Z_train.shape, z_y_train.shape)
# print(Z_val.shape, z_y_val.shape)
y_val = pd.Series(y_img_val, index=cnn_val_ids)
y_train = pd.Series(y_img_train, index=cnn_train_ids)

y_val_aligned = y_val.loc[z_id_val].values
y_train_aligned = y_train.loc[z_id_train].values

preds, y_test = intermediate_fusion(Z_train, y_train_aligned, Z_val, y_val_aligned)


True
True
True
True
F1: 0.26125393285934295
accuracy: 0.6
              precision    recall  f1-score   support

           0       0.24      0.29      0.26        77
           1       0.25      0.34      0.29        41
           2       0.18      0.25      0.21        24
           3       0.20      0.30      0.24        77
           4       0.89      0.77      0.83       463
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00        10

    accuracy                           0.60       700
   macro avg       0.25      0.28      0.26       700
weighted avg       0.66      0.60      0.63       700



##Test + Submission

In [ ]:
test_images = np.load(path + data_path + "test1_images.npy")
test_ids = np.load(path + data_path + "test1_ids.npy", allow_pickle=True)
test_loader = build_testdata(test_images, test_ids)

Test file speciifc late fusion setup
- gets rf and cnn late features (probs)

In [ ]:
def late_fusion_loader(test_images, test_ids, test_loader):

  ########### RF PART ############
  rf_info = joblib.load("rf_pipeline.pkl")
  rf_model = rf_info["model"]
  X_test_raw, rf_ids = load_test_data(data_dir=path + data_path)
  X_test = transform_with_preprocessors(X_test_raw,
                                        rf_info["age_median"],
                                        rf_info["sex_encoder"],
                                        rf_info["site_encoder"])

  rf_probs = rf_model.predict_proba(X_test)

  ########### CNN PART ##############
  test_cnn = CNN(device=DEVICE, num_classes=7).to(DEVICE)
  test_cnn.load_state_dict(torch.load("100epoch_cnn_best.pt", map_location=torch.device(DEVICE)))
  cnn_probs, _, test_ids = predict_test(test_cnn, test_loader)

  print(len(test_ids))

  ########### DF STRUCTURING ############
  cnn_df = pd.DataFrame({"ID":test_ids})
  rf_df = pd.DataFrame({"ID":rf_ids})
  print(rf_probs.shape[1] == cnn_probs.shape[1])
  for i in range(rf_probs.shape[1]):
    cnn_df[f'cnn_prob_{i}'] = cnn_probs[:, i]
    rf_df[f'rf_prob_{i}'] = rf_probs[:, i]


  return rf_df, cnn_df


Test file specific intermediate fusion setup
- gets rf and cnn intermediate features and returns df with fusion predictions

In [ ]:
def inter_fusion_loader(test_images, test_ids, test_loader):

  ######## RF PART ##########
  rf_info = joblib.load("rf_pipeline.pkl")
  rf_model = rf_info["model"]
  X_test_raw, rf_ids = load_test_data(data_dir=path + data_path)
  X_test = transform_with_preprocessors(X_test_raw,
                                        rf_info["age_median"],
                                        rf_info["sex_encoder"],
                                        rf_info["site_encoder"])


  tab_train, tab_val = get_rf_feats(rf_model, X_train, X_test)

  ####### CNN PART ############
  fus_cnn = CNN(device=DEVICE, num_classes=7).to(DEVICE)
  fus_cnn.load_state_dict(torch.load("100epoch_cnn_best.pt", map_location=torch.device(DEVICE)))

  Z_img_test, ids_test = get_features(fus_cnn, test_loader, DEVICE)  # get test features
  print(Z_img_test.shape, ids_test.shape)
  train_rf_df = pd.read_csv(train_rf_save_path)
  # load precomputed train/val features
  data = np.load("cnn_features.npz", allow_pickle=True)

  Z_train = data["Z_img_train"]
  y_img_train = data["y_img_train"]
  ids_train = data["ids_train"]

  # get truncated svd features
  tab_train, tab_test = apply_svd(tab_train, tab_val, rf_train_ids, rf_ids, 'rf_svd')
  img_train, img_test = apply_svd(Z_train, Z_img_test, ids_train, ids_test, 'cnn_svd')

  Z_train, z_train_ids  = fuse_mid_features(tab_train, img_train)
  Z_test, z_test_ids = fuse_mid_features(tab_test, img_test)


  y_train = pd.Series(y_img_train, index=z_train_ids)

  y_train_aligned = y_train.loc[z_train_ids].values

  # train/val split as train for test
  # Z_full = np.vstack([Z_train, Z_val])
  # y_full = np.concatenate([z_y_train, z_y_val])
  preds = intermediate_fusion(Z_train, y_train_aligned, Z_test)

  test_df = pd.DataFrame({'ID': z_test_ids,
                          "fusion_preds": preds})


  return test_df



In [ ]:
def create_sub(df):
  '''creates submission file (works with only 250 test samples)'''
  submission = pd.read_csv(path + data_path + "sample_submission.csv") # load submission file to update [ID, label]

  submission = submission.merge(test_df, how='left', left_on="ID", right_on="ID")
  submission["label"] = submission['fusion_preds'].fillna(0).astype(int) # test1 is 250 rows but submission needs 500 rows to submit to kaggle
  submission = submission[['ID', "label"]]
  print(submission.shape) # should be (500, 2)

  file_path = "Colab Notebooks/cs1851/submission.csv"
  fusion_save_path = os.path.join(os.path.dirname(path), file_path)
  submission.to_csv(fusion_save_path, index=False)
  print(f"\nSaved test fusion predictions to: {fusion_save_path}")

In [ ]:
########## LATE FUSION SUBMISSION ############

rf_df, cnn_df = late_fusion_loader(test_images, test_ids, test_loader)

###### type of fusion #########
# lr = train_lr_fusion(val_rf_df, val_cnn_df, 'rf', 'cnn')   # logistic regression
# test_df = fit_lr_fusion(lr, rf_df, cnn_df, 'rf', 'cnn')

test_df = avg_fusion(rf_df, cnn_df, 'rf', 'cnn')     # average fusion

create_sub(test_df)

250
True
(500, 2)


In [ ]:
########## INTERMEDIATE FUSION SUBMISSION #############
test_df = inter_fusion_loader(test_images, test_ids, test_loader)

create_sub(test_df)


(250, 100352) (250,)
True
True
True
True
(500, 2)

Saved test fusion predictions to: /content/drive/MyDrive/Colab Notebooks/cs1851/submission.csv
